# Lab 4 · FourCastNet으로 48시간 날씨 예측하기

2022년 9월 1일 00시(UTC)부터 이틀 동안 동아시아의 바람과 기압이 어떻게 변했을까요? 사전 학습된 FourCastNet에 전 세계의 기상 자료를 넣어 6시간 뒤의 대기 상태를 예측합니다. 예측한 상태를 다시 입력하는 과정을 여덟 번 반복하면 48시간 예보를 얻습니다.

예보가 끝나면 같은 시각의 ERA5와 비교합니다. ERA5는 실제 관측과 수치모델을 결합해 당시 대기 상태를 추정한 재분석 자료입니다.

[실습 목차](../../00_Start_Here.ipynb) · [이전 실습 · Lab 3 열전도](../03_heat_conduction/Lab_3_Heat_Conduction.ipynb) · [열판 FNO 실습](../../02_FNO.ipynb)

## FourCastNet의 입력과 출력

FourCastNet은 AFNO(Adaptive Fourier Neural Operator) 층을 사용해 푸리에 공간에서 서로 다른 위치의 정보를 결합합니다. 실습 모델의 입력은 전 지구를 덮는 720 × 1440 격자입니다. 격자점마다 지표 부근과 여러 기압면의 기온, 습도, 바람 등 26개 변수의 값이 들어갑니다. 출력은 같은 격자에서 6시간 뒤의 26개 변수입니다.

![FourCastNet의 전 지구 입력 격자와 동아시아 평가 영역](../../ETC/assets/teaching/lab4-domain.svg)

[예보 코드](source_code/run_forecast.py)는 모델의 예측값을 다음 시각의 입력으로 사용합니다. 여덟 번의 추론 동안 모델 가중치는 고정됩니다. [평가 코드](source_code/evaluate_weather.py)는 예보가 끝난 뒤 각 시각의 ERA5 자료를 불러와 오차를 계산합니다.

![예보와 ERA5 평가 과정](../../ETC/assets/teaching/lab4-workflow.svg)

## 1. 모델과 자료 준비하기

Jupyter에서 `AI4Sci PhysicsNeMo 2.2.2 (uv / CUDA)` 커널을 선택하고 아래 셀을 실행하세요. 첫 실행에는 모델 약 301 MB와 ERA5 자료 약 622 MB를 내려받습니다. 다음 실행부터는 캐시에 저장된 파일을 사용합니다.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time
import uuid

import numpy as np
import torch
from IPython.display import Image, Markdown, display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / '00_Start_Here.ipynb').is_file()), None)
if ROOT is None:
    raise RuntimeError('실습 저장소 폴더 안에서 이 노트북을 여세요.')
LAB = ROOT / '01_labs/04_weather_forecasting'
SOURCE = LAB / 'source_code'
ASSETS = LAB / 'assets'
CACHE = Path(os.environ.get('AI4SCI_WEATHER_CACHE',
                            '~/.cache/ai4sci/weather')).expanduser().resolve()
DATA = CACHE / 'data-20220901'
COASTLINE = ASSETS / 'coastline.geojson'
COASTLINE_CREDIT = 'Natural Earth 1:110m coastline (public domain)'
CASE_TIME = '2022-09-01T00:00:00Z'
DEVICE = os.environ.get('AI4SCI_DEVICE', 'auto').strip().lower()
if DEVICE == 'auto':
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE not in {'cpu', 'cuda'}:
    raise ValueError('AI4SCI_DEVICE에 auto, cuda, cpu 중 하나를 지정하세요.')
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA를 사용할 수 없습니다. GPU와 선택한 Jupyter 커널을 확인하세요.')
sys.path.insert(0, str(SOURCE))
RUN_COMPLETED = False
EVALUATION_COMPLETED = False
print('Python:', sys.executable)
print('사용 장치:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU (GPU보다 오래 걸립니다)')
print('캐시 폴더:', CACHE)

def run_script(script, *arguments):
    command = [sys.executable, '-u', str(SOURCE / script), *map(str, arguments)]
    started = time.perf_counter()
    with subprocess.Popen(command, cwd=LAB, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)
    return time.perf_counter() - started

In [ ]:
preparation_seconds = run_script('prepare.py', '--cache-dir', CACHE)
provenance = json.loads((DATA / 'provenance.json').read_text())
assert provenance['initial_time_utc'] == CASE_TIME
print(f'자료 준비 및 캐시 확인: {preparation_seconds:.1f}초')
print('초기 상태와 검증용 ERA5 자료를 준비했습니다.')

`initial.npz`를 열어 시작 시각, 배열 크기, 변수 이름을 확인하세요. 모델은 ERA5의 0.25° 격자에서 남극점에 해당하는 마지막 행을 제외한 720개 위도 행을 사용합니다. 상대습도는 ERA5의 기온·비습·기압으로 계산했습니다. 전처리 내역은 캐시의 `data-20220901/provenance.json`에서 볼 수 있습니다.

`truth.npz`에는 예보를 평가할 때 쓸 ERA5 자료가 들어 있습니다.

In [ ]:
with np.load(DATA / 'initial.npz', allow_pickle=False) as saved:
    initial_state = saved['state']
    print('입력 배열 크기 (변수, 위도, 경도):', initial_state.shape)
    print('예보 시작 시각:', np.datetime_as_string(saved['time'], unit='h'), 'UTC')
    print('변수:', ', '.join(saved['variables'].tolist()))
    print('위도:', saved['latitude'][0], '~', saved['latitude'][-1], '도')
    print('경도:', saved['longitude'][0], '~', saved['longitude'][-1], '도')
del initial_state

## 2. 48시간 예보하기

첫 셀은 모델을 불러와 여덟 시각의 예보를 새 폴더에 저장합니다. 마지막에 표시되는 시간에는 모델을 읽고 결과를 쓰는 시간도 포함됩니다. 다음 셀에서 추론에만 걸린 시간과 GPU 메모리 사용량을 확인하세요.

In [ ]:
RUN_COMPLETED = False
EVALUATION_COMPLETED = False
RUN = LAB / 'outputs' / f'forecast-{uuid.uuid4().hex[:10]}'
forecast_seconds = run_script('run_forecast.py',
                              '--initial', DATA / 'initial.npz',
                              '--model-dir', CACHE / 'model',
                              '--output', RUN, '--device', DEVICE, '--steps', '8')
RUN_COMPLETED = True
print(f'모델 로딩부터 예보 결과 저장까지: {forecast_seconds:.1f}초')

In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError('먼저 위의 예보 셀을 끝까지 실행하세요.')
runtime = json.loads((RUN / 'runtime.json').read_text())
print(f"모델 추론 8회에 걸린 시간: {runtime['inference_seconds']:.2f}초")
if runtime['cuda_peak_allocated_gib'] is not None:
    print(f"PyTorch가 할당한 최대 GPU 메모리: {runtime['cuda_peak_allocated_gib']:.2f} GiB")
print('위 시간과 메모리는 이번 실행에서 측정한 값입니다.')
with np.load(RUN / 'forecast.npz', allow_pickle=False) as saved:
    print('저장한 변수:', ', '.join(saved['variables'].tolist()))
    print('예보 대상 시각 (UTC):', np.datetime_as_string(saved['times'], unit='h').tolist())
print('결과 폴더:', RUN.relative_to(ROOT))

## 3. ERA5 및 지속 예보와 비교하기

시작 시각의 바람·기온·기압이 이틀 동안 그대로라고 가정하면 간단한 예보를 하나 만들 수 있습니다. 이를 지속 예보(persistence forecast)라고 합니다. FourCastNet과 지속 예보 중 어느 쪽이 각 시각의 ERA5에 더 가까웠는지 비교합니다.

표는 전 지구와 동아시아(북위 10–55°, 동경 100–160°)의 평균제곱근오차(RMSE)를 보여 줍니다. 격자점별 오차를 제곱하고 격자 면적에 비례해 가중평균한 뒤 제곱근을 구합니다. 바람은 동서·남북 성분을 함께 계산해 방향과 세기의 차이를 반영합니다. 기압 오차의 단위는 hPa, 기온 오차의 단위는 K입니다. 온도 차이 1 K는 1 °C와 같습니다. 표에는 6시간 뒤부터 48시간 뒤까지의 결과가 나옵니다.

In [ ]:
if not RUN_COMPLETED:
    raise RuntimeError('먼저 위의 예보 셀을 끝까지 실행하세요.')
EVALUATION_COMPLETED = False
EVALUATION = RUN / 'evaluation'
evaluation_seconds = run_script('evaluate_weather.py',
                                '--forecast', RUN / 'forecast.npz',
                                '--truth', DATA / 'truth.npz',
                                '--output-dir', EVALUATION,
                                '--coastline', COASTLINE,
                                '--coastline-attribution', COASTLINE_CREDIT,
                                '--acceptance', ASSETS / 'weather-acceptance.json', '--gif')
summary = json.loads((EVALUATION / 'evaluation.json').read_text())
EVALUATION_COMPLETED = True
print(f'오차 계산, 지도 및 애니메이션 생성: {evaluation_seconds:.1f}초')
print('검증 자료:', summary['reference_kind'])
checks = summary['acceptance']['checks']
print(f"미리 정한 검증 기준 중 충족한 항목: {sum(check['passed'] for check in checks)}/{len(checks)}")
print('아래에서 예보 오차가 큰 시각과 지역을 확인하세요.')

In [ ]:
if not EVALUATION_COMPLETED:
    raise RuntimeError('먼저 위의 검증 셀을 끝까지 실행하세요.')
for region, records in summary['metrics'].items():
    rows = [
        '### ' + {'global': '전 지구', 'east_asia': '동아시아'}.get(region, region), '',
        '각 칸은 **FourCastNet 예보 RMSE / 지속 예보 RMSE** 순서입니다.', '',
        '| 예측 시점 | 바람 벡터 (m/s) | 평균 해면기압 (hPa) | 2 m 기온 (K) |',
        '|---:|---:|---:|---:|',
    ]
    for row in records:
        wind = row['wind_10m']
        pressure = row['variables']['msl']
        temperature = row['variables']['t2m']
        rows.append(
            f"| {row['lead_hours']}시간 뒤 "
            f"| {wind['vector_rmse']:.2f} / {wind['persistence_vector_rmse']:.2f} "
            f"| {pressure['display_rmse']:.2f} / {pressure['display_persistence_rmse']:.2f} "
            f"| {temperature['rmse']:.2f} / {temperature['persistence_rmse']:.2f} |"
        )
    display(Markdown('\n'.join(rows)))
display(Image(filename=str(EVALUATION / 'rmse-by-lead-global.png')))
display(Image(filename=str(EVALUATION / 'rmse-by-lead-east_asia.png')))

## 4. 동아시아 기상 지도 읽기

`variable='wind_speed'`로 두고 `lead_hours`를 6, 24, 48로 바꿔 가며 아래 셀을 실행하세요. 왼쪽은 ERA5, 가운데는 FourCastNet 예보입니다. 두 지도는 여덟 시각에 걸쳐 같은 색 눈금을 씁니다. 오른쪽은 예보에서 ERA5를 뺀 값으로, 0을 중심으로 한 별도 색 눈금을 씁니다.

바람이 가장 강한 곳을 골라 해안선과의 거리, 풍속, 화살표 방향을 비교해 보세요. `msl`을 선택하면 평균 해면기압, `t2m`을 선택하면 지상 2 m 기온을 볼 수 있습니다.

In [ ]:
import matplotlib.pyplot as plt
from evaluate_weather import (
    PLOT_UNITS, draw_map, geographic_lines, load_pair, regional_plot_data,
)

variable = 'wind_speed'  # 'wind_speed', 'msl', 't2m', 'u10m', 'v10m' 중 하나
lead_hours = 24          # 6, 12, 18, 24, 30, 36, 42, 48 중 하나

if not EVALUATION_COMPLETED:
    raise RuntimeError('먼저 위의 검증 셀을 끝까지 실행하세요.')
if variable not in PLOT_UNITS or lead_hours not in range(6, 49, 6):
    raise ValueError('주석에 있는 변수와 6~48시간 사이의 6시간 단위 예측 시점을 선택하세요.')
data = load_pair(RUN / 'forecast.npz', DATA / 'truth.npz')
plot = regional_plot_data(data, variable)
frame = data['lead_hours'].tolist().index(lead_hours)
lines = geographic_lines(COASTLINE)
field_names = {'wind_speed': '10 m wind speed', 'msl': 'Mean sea-level pressure',
               't2m': '2 m temperature', 'u10m': '10 m eastward wind',
               'v10m': '10 m northward wind'}

with plt.rc_context({'font.size': 12, 'axes.titlesize': 14}):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    for ax, key, title in zip(axes, ('reference', 'prediction', 'error'),
                              ('ERA5 reanalysis', 'FourCastNet forecast', 'Forecast minus ERA5')):
        is_error = key == 'error'
        limits = (-plot['error_limit'], plot['error_limit']) if is_error else plot['limits']
        vector = None
        if variable == 'wind_speed':
            vector = (plot['prediction_uv'][frame] - plot['reference_uv'][frame]
                      if is_error else plot[key + '_uv'][frame])
        image = draw_map(ax, plot[key][frame], plot['latitude'], plot['longitude'],
                         limits, 'RdBu_r' if is_error else plot['cmap'], lines, vector)
        ax.set_title(title)
        fig.colorbar(image, ax=ax, orientation='horizontal',
                     label=PLOT_UNITS[variable] + (' difference' if is_error else ''))
    valid_time = np.datetime_as_string(data['times'][frame], unit='h')
    fig.suptitle(f'{field_names[variable]} | +{lead_hours} h | {valid_time} UTC', fontsize=16)
    plt.show()
print(COASTLINE_CREDIT)
del data, plot

### 바람 분포의 변화

애니메이션에서 강풍대가 시간에 따라 이동하는 모습을 확인하세요. ERA5와 예보에서 강풍대의 위치가 처음으로 어긋나는 시각은 언제인가요? 화면에 표시되는 시각은 UTC입니다.

In [ ]:
if not EVALUATION_COMPLETED:
    raise RuntimeError('먼저 위의 검증 셀을 실행해 애니메이션을 만드세요.')
display(Image(filename=str(EVALUATION / 'east-asia-wind-6-to-48h.gif')))

## 결과 해석

1. 24시간과 48시간 뒤, 어느 변수에서 FourCastNet과 지속 예보의 RMSE 차이가 가장 뚜렷했나요? 변수별 단위가 다르므로 같은 변수 안에서 두 예보를 비교하세요.
2. 동아시아 전체 RMSE와 지도에서 읽은 국지적인 오차는 어떤 차이가 있나요? 저기압 중심이나 강풍대를 골라 비교해 보세요.
3. 예보 시간이 길어질수록 오차가 계속 커졌나요? 변수 하나를 골라 여덟 시각의 결과를 확인하세요.

여덟 예보 시각은 모두 같은 초기 상태에서 출발했습니다. 다른 계절이나 기상 상황에서도 비슷한 성능이 나오는지 알려면 시작 날짜를 바꿔 여러 사례를 비교해야 합니다.

## 자료와 출처

- **FourCastNet.** NVIDIA의 26채널 AFNO 모델입니다. 실습은 `nvidia/fourcastnet1`의 고정 버전에서 가중치와 정규화 계수를 받습니다. [모델 카드와 라이선스](https://huggingface.co/nvidia/fourcastnet1)
- **Earth2Studio.** 변수 순서, 격자와 정규화 방식은 NVIDIA의 FCN 구현을 따릅니다. [FCN 코드](https://github.com/NVIDIA/earth2studio/blob/486c5daa98841b0ce93cb78cb70d0dee7a8a56e6/earth2studio/models/px/fcn.py)
- **ERA5.** ECMWF가 운영하는 코페르니쿠스 기후변화서비스의 재분석 자료를 Google 공개 아카이브에서 받습니다. [ARCO-ERA5](https://github.com/google-research/arco-era5)
- **해안선.** 지도에는 Natural Earth의 1:110m 퍼블릭 도메인 해안선을 사용합니다. [Natural Earth](https://www.naturalearthdata.com/about/terms-of-use/)

모델 버전, 자료 이용 조건, 전처리 방식은 [기상 자료 안내](assets/README.md)에 정리돼 있습니다.

[실습 목차](../../00_Start_Here.ipynb) · [이전 실습 · Lab 3 열전도](../03_heat_conduction/Lab_3_Heat_Conduction.ipynb) · [열판 FNO 실습](../../02_FNO.ipynb)